# Basic Machine Learning Model - Philipp

In [1]:
# Import the relevant modules and classes
from prepare_data import prepare_data
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [2]:
start_date = datetime(2015, 1, 1)
end_date = datetime(2022, 12, 31)

hour_memory = 24 * 7 
future_horizon = 168 #! needs to be fixed to 168 (one week) as no further data was available for the original dataset either.

X, y = prepare_data(start_date=start_date, end_date=end_date, hour_memory=hour_memory, future_horizon=future_horizon)

# Ensure X and y have the same index


print("X shape:", X.shape)
print("y shape:", y.shape)

print("X index:", X.index)
print("y index:", y.index)

print("Same index:", X.index.equals(y.index))
print("Same number of rows:", len(X) == len(y))

# Split the data into training, validation, and test sets based on the starttime column
train_val_mask = X['starttime'] < datetime(2022, 1, 1)
test_mask = X['starttime'] >= datetime(2022, 1, 1)

X_train_val = X.loc[train_val_mask].copy()
y_train_val = y.loc[train_val_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


folds = [
    {
        'train_end': '2018-01-01',
        'val_start': '2018-01-01',
        'val_end': '2019-01-01'
    },
    {
        'train_end': '2019-01-01',
        'val_start': '2019-01-01',
        'val_end': '2020-01-01'
    },
    {
        'train_end': '2020-01-01',
        'val_start': '2020-01-01',
        'val_end': '2021-01-01'
    },
    {
        'train_end': '2021-01-01',
        'val_start': '2021-01-01',
        'val_end': '2022-01-01'
    }
]

train_folds = []

for fold in folds:

    train_mask = (
        X['starttime'] < pd.Timestamp(fold['train_end'])
    )

    val_mask = (
        (X['starttime'] >= pd.Timestamp(fold['val_start'])) &
        (X['starttime'] < pd.Timestamp(fold['val_end']))
    )

    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]

    X_val = X.loc[val_mask]
    y_val = y.loc[val_mask]

    train_folds.append((X_train, y_train, X_val, y_val))

    print(
        f"Train: {X_train['starttime'].min()} "
        f"to {X_train['starttime'].max()}"
    )

    print(
        f"Validation: {X_val['starttime'].min()} "
        f"to {X_val['starttime'].max()}"
    )

X shape: (2908, 6862)
y shape: (2908, 2352)
X index: RangeIndex(start=0, stop=2908, step=1)
y index: RangeIndex(start=0, stop=2908, step=1)
Same index: True
Same number of rows: True
Train: 2015-01-08 00:00:00 to 2017-12-31 00:00:00
Validation: 2018-01-01 00:00:00 to 2018-12-31 00:00:00
Train: 2015-01-08 00:00:00 to 2018-12-31 00:00:00
Validation: 2019-01-01 00:00:00 to 2019-12-31 00:00:00
Train: 2015-01-08 00:00:00 to 2019-12-31 00:00:00
Validation: 2020-01-01 00:00:00 to 2020-12-31 00:00:00
Train: 2015-01-08 00:00:00 to 2020-12-31 00:00:00
Validation: 2021-01-01 00:00:00 to 2021-12-31 00:00:00


In [3]:
# Train the model:

#TODO: Implement the model here